# AI-Based Hospital Readmission Prediction System

# Model Training

# Objective

The objective of this notebook is to train and tune multiple machine
learning classification models for predicting 30-day hospital readmission.

# Models

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting

# Training Strategy

- Load the feature-engineered dataset
- Separate features and target
- Perform stratified train-test split
- Build preprocessing pipelines
- Train baseline models
- Perform cross-validation
- Tune selected models using RandomizedSearchCV
- Save trained models and training data

# Important Note

The final model will NOT be selected in this notebook.
Final model selection will be performed in Notebook 06 using
held-out test-set evaluation.

# Import Libraries

In [1]:
# Import os for creating folders and handling file paths
import os

# Import joblib for saving trained machine learning models
import joblib

# Import numpy for numerical operations
import numpy as np

# Import pandas for loading and manipulating datasets
import pandas as pd

# Import Logistic Regression model
from sklearn.linear_model import LogisticRegression

# Import Decision Tree classifier
from sklearn.tree import DecisionTreeClassifier

# Import Random Forest classifier
from sklearn.ensemble import RandomForestClassifier

# Import Gradient Boosting classifier
from sklearn.ensemble import GradientBoostingClassifier

# Import train_test_split for creating training and testing datasets
from sklearn.model_selection import train_test_split

# Import StratifiedKFold for stratified cross-validation
from sklearn.model_selection import StratifiedKFold

# Import RandomizedSearchCV for hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# Import preprocessing components
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Import OrdinalEncoder for the Gradient Boosting pipeline
from sklearn.preprocessing import OrdinalEncoder

# Import F1 scoring because the target is imbalanced
from sklearn.metrics import make_scorer
from sklearn.metrics import f1_score

# Display a message confirming that libraries were imported
print("All required libraries imported successfully.")

All required libraries imported successfully.


# 1. Load Feature-Engineered Dataset

The feature-engineered dataset created in Notebook 04 will be used
for model training.

The original raw dataset is not modified.

In [2]:
# Define the path of the feature-engineered dataset
data_path = "artifacts/feature_engineered_data.csv"

# Load the feature-engineered dataset
df = pd.read_csv(data_path)

# Display the first five rows
df.head()

C:\Users\Asus\AppData\Local\Temp\ipykernel_5044\1518874620.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted_binary,total_prior_visits,total_procedures,medications_per_diagnosis
0,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,0,0,41,1.000000
1,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,...,No,No,No,No,Ch,Yes,0,0,59,2.000000
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,...,No,No,No,No,No,Yes,0,3,16,2.166667
3,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,...,No,No,No,No,Ch,Yes,0,0,45,2.285714
4,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,...,No,No,No,No,Ch,Yes,0,0,51,1.600000


# Check Dataset Shape

In [3]:
# Display the number of rows in the feature-engineered dataset
print("Number of Rows:", df.shape[0])

# Display the number of columns in the feature-engineered dataset
print("Number of Columns:", df.shape[1])

Number of Rows: 101766
Number of Columns: 49


# Check Target Distribution

In [4]:
# Count the number of observations in each target class
target_counts = df["readmitted_binary"].value_counts().sort_index()

# Display the target distribution
print("Target Distribution:")
print(target_counts)

# Calculate the percentage of each target class
target_percentage = (
    df["readmitted_binary"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

# Display the target percentages
print("\nTarget Percentage:")
print(target_percentage.round(2))

Target Distribution:
readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

Target Percentage:
readmitted_binary
0    88.84
1    11.16
Name: proportion, dtype: float64


# 2. Separate Features and Target

The target variable is:

`readmitted_binary`

All remaining columns are treated as input features.

The target is not included in the feature matrix to prevent target
leakage during model training.

# Separate X and y

In [5]:
# Create X containing all input features
X = df.drop(columns=["readmitted_binary"])

# Create y containing only the binary target
y = df["readmitted_binary"]

# Display the shape of the feature matrix
print("Feature Matrix Shape:", X.shape)

# Display the shape of the target vector
print("Target Shape:", y.shape)

Feature Matrix Shape: (101766, 48)
Target Shape: (101766,)


# Identify Feature Types

In [6]:
# Identify numerical columns from the feature matrix
numerical_columns = X.select_dtypes(
    include=np.number
).columns.tolist()

# Identify categorical columns from the feature matrix
categorical_columns = X.select_dtypes(
    exclude=np.number
).columns.tolist()

# Display the number of numerical columns
print("Number of Numerical Features:", len(numerical_columns))

# Display the number of categorical columns
print("Number of Categorical Features:", len(categorical_columns))

Number of Numerical Features: 14
Number of Categorical Features: 34


# Display Feature Names

In [7]:
# Display all numerical feature names
print("Numerical Features:")
print(numerical_columns)

# Display all categorical feature names
print("\nCategorical Features:")
print(categorical_columns)

Numerical Features:
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'total_prior_visits', 'total_procedures', 'medications_per_diagnosis']

Categorical Features:
['race', 'gender', 'age', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


# 3. Train-Test Split

The dataset will be divided into:

- 80% Training Data
- 20% Testing Data

Stratification is used because the target variable is imbalanced.

The test set will remain untouched until Notebook 06.

# Train-Test Split

In [8]:
# Split the dataset into training and testing sets
# Stratification preserves the target class distribution
# random_state=42 ensures reproducible results

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the shape of the training features
print("Training Feature Shape:", X_train.shape)

# Display the shape of the testing features
print("Testing Feature Shape:", X_test.shape)

# Display the shape of the training target
print("Training Target Shape:", y_train.shape)

# Display the shape of the testing target
print("Testing Target Shape:", y_test.shape)

Training Feature Shape: (81412, 48)
Testing Feature Shape: (20354, 48)
Training Target Shape: (81412,)
Testing Target Shape: (20354,)


# Verify Stratification

In [9]:
# Calculate the positive-class percentage in the complete dataset
overall_rate = y.mean() * 100

# Calculate the positive-class percentage in the training dataset
train_rate = y_train.mean() * 100

# Calculate the positive-class percentage in the testing dataset
test_rate = y_test.mean() * 100

# Display the percentages for comparison
print("Overall 30-Day Readmission Rate:", round(overall_rate, 2), "%")
print("Training 30-Day Readmission Rate:", round(train_rate, 2), "%")
print("Testing 30-Day Readmission Rate:", round(test_rate, 2), "%")

Overall 30-Day Readmission Rate: 11.16 %
Training 30-Day Readmission Rate: 11.16 %
Testing 30-Day Readmission Rate: 11.16 %


# 4. Numerical Preprocessing Pipeline

For numerical features:

1. Missing values will be replaced using the median.
2. Features will be standardized using StandardScaler.

The preprocessing will be fitted only on the training data through
the machine-learning pipeline.

# Numerical Pipeline

In [10]:
# Create a pipeline for numerical features
numerical_pipeline = Pipeline(
    steps=[
        # Replace missing numerical values with the median
        ("imputer", SimpleImputer(strategy="median")),

        # Standardize numerical features
        ("scaler", StandardScaler())
    ]
)

# Display the numerical preprocessing pipeline
numerical_pipeline

,steps,"[('imputer', ...), ('scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


# 5. Categorical Preprocessing Pipeline

For categorical features:

1. Missing values will be replaced using the most frequent category.
2. Categories will be converted into one-hot encoded features.
3. Unknown categories in test data will be ignored.

# Categorical Pipeline

In [11]:
# Create a pipeline for categorical features
categorical_pipeline = Pipeline(
    steps=[
        # Replace missing categorical values with the most frequent value
        ("imputer", SimpleImputer(strategy="most_frequent")),

        # Convert categorical values into one-hot encoded features
        # handle_unknown='ignore' prevents errors for unseen categories
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ]
)

# Display the categorical preprocessing pipeline
categorical_pipeline

,steps,"[('imputer', ...), ('onehot', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,categories,'auto'


# Combined Preprocessor

In [12]:
# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        # Apply numerical pipeline to numerical columns
        ("num", numerical_pipeline, numerical_columns),

        # Apply categorical pipeline to categorical columns
        ("cat", categorical_pipeline, categorical_columns)
    ]
)

# Display the combined preprocessing object
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


# 6. Logistic Regression Baseline

Logistic Regression is used as the baseline classification model.

Class weights are balanced because the positive class
(30-day readmission) is relatively less frequent.

# Logistic Regression Pipeline

In [13]:
# Create the Logistic Regression model
logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Create a complete pipeline containing preprocessing and Logistic Regression
logistic_pipeline = Pipeline(
    steps=[
        # Apply preprocessing before model training
        ("preprocessor", preprocessor),

        # Train the Logistic Regression model
        ("model", logistic_model)
    ]
)

# Display the Logistic Regression pipeline
logistic_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Train Logistic Regression

In [14]:
# Train the Logistic Regression pipeline using only training data
logistic_pipeline.fit(X_train, y_train)

# Display a confirmation message
print("Logistic Regression training completed.")

Logistic Regression training completed.


# 7. Decision Tree Model

Decision Tree is a non-linear classification model.

Class weights are balanced to give additional importance to
the minority readmission class.

# Decision Tree Pipeline

In [15]:
# Create the Decision Tree model
decision_tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

# Create a pipeline containing preprocessing and Decision Tree
decision_tree_pipeline = Pipeline(
    steps=[
        # Apply preprocessing before model training
        ("preprocessor", preprocessor),

        # Train the Decision Tree model
        ("model", decision_tree_model)
    ]
)

# Display the Decision Tree pipeline
decision_tree_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Train Decision Tree

In [16]:
# Train the Decision Tree pipeline using the training dataset
decision_tree_pipeline.fit(X_train, y_train)

# Display a confirmation message
print("Decision Tree training completed.")

Decision Tree training completed.


# 8. Random Forest Model

Random Forest combines multiple decision trees.

It can capture non-linear relationships and interactions
between features.

Class weights are balanced to address target imbalance.

# Random Forest Pipeline

In [17]:
# Create the Random Forest model
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Create a pipeline containing preprocessing and Random Forest
random_forest_pipeline = Pipeline(
    steps=[
        # Apply preprocessing before model training
        ("preprocessor", preprocessor),

        # Train the Random Forest model
        ("model", random_forest_model)
    ]
)

# Display the Random Forest pipeline
random_forest_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Train Random Forest

In [20]:
# Train the Random Forest pipeline using the training dataset
random_forest_pipeline.fit(X_train, y_train)

# Display a confirmation message
print("Random Forest training completed.")

Random Forest training completed.


# 9. Gradient Boosting Model

Gradient Boosting builds models sequentially to reduce prediction errors.

For this model, categorical variables are ordinal encoded.

Missing categorical values are handled using the most frequent category.

# Gradient Boosting Preprocessor

In [21]:
# Create a separate numerical preprocessing pipeline for Gradient Boosting
gb_numerical_pipeline = Pipeline(
    steps=[
        # Replace missing numerical values with the median
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Create a categorical preprocessing pipeline for Gradient Boosting
gb_categorical_pipeline = Pipeline(
    steps=[
        # Replace missing categorical values with the most frequent value
        ("imputer", SimpleImputer(strategy="most_frequent")),

        # Convert categorical values into numerical ordinal values
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ]
)

# Combine numerical and categorical preprocessing
gb_preprocessor = ColumnTransformer(
    transformers=[
        # Apply numerical preprocessing
        ("num", gb_numerical_pipeline, numerical_columns),

        # Apply categorical preprocessing
        ("cat", gb_categorical_pipeline, categorical_columns)
    ]
)

# Display the Gradient Boosting preprocessor
gb_preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


# Gradient Boosting Model

In [22]:
# Create the Gradient Boosting classifier
gradient_boosting_model = GradientBoostingClassifier(
    random_state=42
)

# Create the complete Gradient Boosting pipeline
gradient_boosting_pipeline = Pipeline(
    steps=[
        # Apply preprocessing before model training
        ("preprocessor", gb_preprocessor),

        # Train the Gradient Boosting model
        ("model", gradient_boosting_model)
    ]
)

# Display the Gradient Boosting pipeline
gradient_boosting_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Train Gradient Boosting

In [23]:
# Train the Gradient Boosting pipeline using the training dataset
gradient_boosting_pipeline.fit(X_train, y_train)

# Display a confirmation message
print("Gradient Boosting training completed.")

Gradient Boosting training completed.


# 10. Cross-Validation Setup

Stratified 3-fold cross-validation will be used.

F1-score is selected as the primary cross-validation metric because
the target variable is imbalanced and both precision and recall
are important.

# Create Cross-Validation Strategy

In [24]:
# Create a stratified 3-fold cross-validation strategy
# shuffle=True randomly distributes observations while preserving
# the class distribution in each fold
cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Create an F1 scoring object
f1_scorer = make_scorer(f1_score)

# Display the cross-validation configuration
print("Cross-Validation: 3 Stratified Folds")
print("Scoring Metric: F1 Score")

Cross-Validation: 3 Stratified Folds
Scoring Metric: F1 Score


# 11. Hyperparameter Tuning — Logistic Regression

RandomizedSearchCV will search through different hyperparameter
combinations.

The best configuration will be selected using cross-validated F1-score.

# Logistic Regression Parameter Search

In [25]:
# Define hyperparameter values for Logistic Regression
logistic_parameters = {
    # Test different regularization strengths
    "model__C": [0.001, 0.01, 0.1, 1, 10],

    # Test different solvers
    "model__solver": ["liblinear"]
}

# Create RandomizedSearchCV for Logistic Regression
logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=logistic_parameters,
    n_iter=5,
    scoring=f1_scorer,
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    refit=True
)

# Display a confirmation message
print("Logistic Regression hyperparameter search created.")

Logistic Regression hyperparameter search created.


# Tune Logistic Regression

In [26]:
# Run hyperparameter tuning using only the training dataset
logistic_search.fit(X_train, y_train)

# Display the best hyperparameters found during cross-validation
print("Best Logistic Regression Parameters:")
print(logistic_search.best_params_)

# Display the best cross-validated F1-score
print(
    "Best Logistic Regression CV F1:",
    round(logistic_search.best_score_, 4)
)

Best Logistic Regression Parameters:
{'model__solver': 'liblinear', 'model__C': 0.01}
Best Logistic Regression CV F1: 0.2582


# 12. Hyperparameter Tuning — Decision Tree

# Decision Tree Parameter Search

In [27]:
# Define hyperparameter ranges for Decision Tree
decision_tree_parameters = {
    # Test different maximum tree depths
    "model__max_depth": [3, 5, 7, 10, 15, 20, None],

    # Test different minimum samples required to split a node
    "model__min_samples_split": [2, 5, 10],

    # Test different minimum samples required at a leaf
    "model__min_samples_leaf": [1, 2, 5, 10]
}

# Create RandomizedSearchCV for Decision Tree
decision_tree_search = RandomizedSearchCV(
    estimator=decision_tree_pipeline,
    param_distributions=decision_tree_parameters,
    n_iter=10,
    scoring=f1_scorer,
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    refit=True
)

# Display confirmation
print("Decision Tree hyperparameter search created.")

Decision Tree hyperparameter search created.


# Tune Decision Tree

In [28]:
# Run hyperparameter tuning using only training data
decision_tree_search.fit(X_train, y_train)

# Display the best Decision Tree parameters
print("Best Decision Tree Parameters:")
print(decision_tree_search.best_params_)

# Display the best cross-validated F1-score
print(
    "Best Decision Tree CV F1:",
    round(decision_tree_search.best_score_, 4)
)

Best Decision Tree Parameters:
{'model__min_samples_split': 5, 'model__min_samples_leaf': 10, 'model__max_depth': 5}
Best Decision Tree CV F1: 0.2601


# 13. Hyperparameter Tuning — Random Forest

# Random Forest Parameter Search

In [29]:
# Define hyperparameter ranges for Random Forest
random_forest_parameters = {
    # Test different numbers of trees
    "model__n_estimators": [100, 200, 300],

    # Test different maximum tree depths
    "model__max_depth": [None, 10, 15, 20, 30],

    # Test different minimum samples required to split
    "model__min_samples_split": [2, 5, 10],

    # Test different minimum samples required at a leaf
    "model__min_samples_leaf": [1, 2, 5]
}

# Create RandomizedSearchCV for Random Forest
random_forest_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=random_forest_parameters,
    n_iter=10,
    scoring=f1_scorer,
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    refit=True
)

# Display confirmation
print("Random Forest hyperparameter search created.")

Random Forest hyperparameter search created.


# Tune Random Forest

In [30]:
# Run Random Forest hyperparameter tuning using training data
random_forest_search.fit(X_train, y_train)

# Display the best Random Forest parameters
print("Best Random Forest Parameters:")
print(random_forest_search.best_params_)

# Display the best cross-validated F1-score
print(
    "Best Random Forest CV F1:",
    round(random_forest_search.best_score_, 4)
)

Best Random Forest Parameters:
{'model__n_estimators': 300, 'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_depth': 30}
Best Random Forest CV F1: 0.2666


# 14. Prepare Final Training Models

The tuned models obtained from RandomizedSearchCV will be retained.

Gradient Boosting will use its trained baseline configuration.

The final model will be selected later using the held-out test set.

# Store Models

In [31]:
# Store the baseline and tuned models in a dictionary
trained_models = {
    # Store the trained baseline Logistic Regression model
    "Logistic Regression": logistic_pipeline,

    # Store the tuned Logistic Regression model
    "Tuned Logistic Regression": logistic_search.best_estimator_,

    # Store the baseline Decision Tree model
    "Decision Tree": decision_tree_pipeline,

    # Store the tuned Decision Tree model
    "Tuned Decision Tree": decision_tree_search.best_estimator_,

    # Store the baseline Random Forest model
    "Random Forest": random_forest_pipeline,

    # Store the tuned Random Forest model
    "Tuned Random Forest": random_forest_search.best_estimator_,

    # Store the Gradient Boosting model
    "Gradient Boosting": gradient_boosting_pipeline
}

# Display all stored model names
print("Models Available for Evaluation:")
for model_name in trained_models:
    print("-", model_name)

Models Available for Evaluation:
- Logistic Regression
- Tuned Logistic Regression
- Decision Tree
- Tuned Decision Tree
- Random Forest
- Tuned Random Forest
- Gradient Boosting


# Create Artifacts Folder

In [32]:
# Define the artifacts directory
artifacts_folder = "artifacts"

# Create the directory if it does not already exist
os.makedirs(
    artifacts_folder,
    exist_ok=True
)

# Display confirmation
print("Artifacts folder is ready.")

Artifacts folder is ready.


# Save Models

In [33]:
# Save the baseline Logistic Regression pipeline
joblib.dump(
    logistic_pipeline,
    "artifacts/logistic_regression_pipeline.pkl"
)

# Save the baseline Decision Tree pipeline
joblib.dump(
    decision_tree_pipeline,
    "artifacts/decision_tree_pipeline.pkl"
)

# Save the baseline Random Forest pipeline
joblib.dump(
    random_forest_pipeline,
    "artifacts/random_forest_pipeline.pkl"
)

# Save the Gradient Boosting pipeline
joblib.dump(
    gradient_boosting_pipeline,
    "artifacts/gradient_boosting_pipeline.pkl"
)

# Save the tuned Logistic Regression model
joblib.dump(
    logistic_search.best_estimator_,
    "artifacts/tuned_logistic_regression_pipeline.pkl"
)

# Save the tuned Decision Tree model
joblib.dump(
    decision_tree_search.best_estimator_,
    "artifacts/tuned_decision_tree_pipeline.pkl"
)

# Save the tuned Random Forest model
joblib.dump(
    random_forest_search.best_estimator_,
    "artifacts/tuned_random_forest_pipeline.pkl"
)

# Display confirmation
print("All trained models saved successfully.")

All trained models saved successfully.


# Save Train/Test Data

In [34]:
# Save the training feature data for reproducibility
joblib.dump(
    X_train,
    "artifacts/model_X_train.pkl"
)

# Save the testing feature data for reproducibility
joblib.dump(
    X_test,
    "artifacts/model_X_test.pkl"
)

# Save the training target
joblib.dump(
    y_train,
    "artifacts/model_y_train.pkl"
)

# Save the testing target
joblib.dump(
    y_test,
    "artifacts/model_y_test.pkl"
)

# Display confirmation
print("Training and testing datasets saved successfully.")

Training and testing datasets saved successfully.


# Save Model Input Columns

In [35]:
# Save the feature column names used by the models
model_input_columns = X.columns.tolist()

# Save the feature column names as a pickle file
joblib.dump(
    model_input_columns,
    "artifacts/model_input_columns.pkl"
)

# Display the number of saved input columns
print(
    "Number of Model Input Columns:",
    len(model_input_columns)
)

Number of Model Input Columns: 48


# Verify Saved Models

In [36]:
# Create a list containing all expected model files
model_files = [
    "logistic_regression_pipeline.pkl",
    "decision_tree_pipeline.pkl",
    "random_forest_pipeline.pkl",
    "gradient_boosting_pipeline.pkl",
    "tuned_logistic_regression_pipeline.pkl",
    "tuned_decision_tree_pipeline.pkl",
    "tuned_random_forest_pipeline.pkl"
]

# Check each model file
for file_name in model_files:

    # Create the complete file path
    file_path = os.path.join(
        artifacts_folder,
        file_name
    )

    # Check whether the file exists
    print(
        file_name,
        "->",
        os.path.exists(file_path)
    )

logistic_regression_pipeline.pkl -> True
decision_tree_pipeline.pkl -> True
random_forest_pipeline.pkl -> True
gradient_boosting_pipeline.pkl -> True
tuned_logistic_regression_pipeline.pkl -> True
tuned_decision_tree_pipeline.pkl -> True
tuned_random_forest_pipeline.pkl -> True


# 15. Model Training Summary

The following machine learning models have been trained:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting

Hyperparameter tuning was performed for:

- Logistic Regression
- Decision Tree
- Random Forest

Stratified 3-fold cross-validation with F1-score was used for tuning.

The held-out test set was not used for model selection during training.

Final model selection will be performed in Notebook 06 using:

- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix

# Final Completion Check

In [37]:
# Display a separator for readability
print("==========================================")

# Display notebook completion message
print("NOTEBOOK 05 - MODEL TRAINING COMPLETED")

# Display the number of training observations
print("Training Samples:", len(X_train))

# Display the number of testing observations
print("Testing Samples:", len(X_test))

# Display the number of trained models
print("Models Trained:", len(trained_models))

# Display the number of tuned models
print("Models Tuned:", 3)

# Display a reminder that final selection happens in Notebook 06
print("Final model selection: Notebook 06")

# Display closing separator
print("==========================================")

NOTEBOOK 05 - MODEL TRAINING COMPLETED
Training Samples: 81412
Testing Samples: 20354
Models Trained: 7
Models Tuned: 3
Final model selection: Notebook 06
